# Step 1 — Fast Tuning Weekly Loop

This notebook runs your 3 weight sweeps, computes composite uplift, ranks winners, and appends results to a persistent leaderboard with a final `run_notes` column.

## What this notebook does (ML view)
This notebook evaluates three score-fusion weight settings and picks the winner by **composite uplift**:
- $\Delta\text{topk\_target}$
- $\Delta\text{ndcg@k}$
- Composite uplift = $\Delta\text{topk\_target} + \Delta\text{ndcg@k}$

### Why these metrics
- **Top-k target quality** checks whether the highest-ranked candidate slices are actually better (on average) than alternatives.
- **NDCG@k** rewards getting the *best* items near the top of the ranked list (position-sensitive ranking quality).

### Why weighted fusion
The evaluator combines three signals into one composite score:
1. model score (normalized),
2. periodicity score,
3. drum alignment score.

This is a standard linear score-fusion pattern in ranking systems:
$$
s_{composite} = w_m\,\hat{s}_{model} + w_p\,s_{periodicity} + w_d\,s_{drum}, \quad w_m+w_p+w_d=1
$$
The weekly loop is: run, compare deltas, rank, keep notes, and lock the top setting until gains flatten.

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from datetime import datetime
from pathlib import Path

import pandas as pd

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

## Code logic walkthrough
1. **Imports + display config**: makes result tables readable in notebook output.
2. **Run configuration cell**: centralizes file paths, bars, top-k, and the weight grid.
3. **`run_one_sweep(...)`**: executes `evaluate_loop_models.py` once per weight tuple and writes rows/metrics artifacts.
4. **`parse_metrics(...)`**: loads JSON output and extracts comparable deltas.
5. **Run table cell**: computes per-run ranking and keeps `run_notes` as the final column.
6. **Leaderboard cell**: appends each run to historical CSV and re-ranks all-time winners.
7. **Optional notes update**: lets you revise notes after listening/QC without rerunning sweeps.

This structure keeps experimentation reproducible: all decisions become rows in a persistent leaderboard.

In [ ]:
# --- Configure once per run ---
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()

EVAL_SCRIPT = PROJECT_ROOT / 'evaluate_loop_models.py'
AUDIO_FOLDER = PROJECT_ROOT / 'data' / 'exported_loops'
MODEL_A = PROJECT_ROOT / 'training' / 'models' / 'loop_ranker_drums_only.json'
FEATURES_A = PROJECT_ROOT / 'training' / 'models' / 'features_drums_only.json'

TOP_K = 5
BARS = '4,8'
RUN_NOTE = 'weekly sweep run'  # <- edit per run

WEIGHT_CONFIGS = [
    (0.75, 0.15, 0.10),
    (0.65, 0.20, 0.15),
    (0.80, 0.10, 0.10),
]

OUT_DIR = PROJECT_ROOT / 'training' / 'models' / 'fast_tune_weight_sweeps'
OUT_DIR.mkdir(parents=True, exist_ok=True)
LEADERBOARD_CSV = OUT_DIR / 'weekly_weight_leaderboard.csv'

assert EVAL_SCRIPT.exists(), f'Missing evaluator: {EVAL_SCRIPT}'
assert AUDIO_FOLDER.exists(), f'Missing audio folder: {AUDIO_FOLDER}'
assert MODEL_A.exists(), f'Missing model file: {MODEL_A}'
assert FEATURES_A.exists(), f'Missing feature list: {FEATURES_A}'

print('Project root:', PROJECT_ROOT)
print('Output dir   :', OUT_DIR)

In [ ]:
def run_one_sweep(weight_model: float, weight_periodicity: float, weight_drum: float) -> Path:
    tag = f"{weight_model:.2f}_{weight_periodicity:.2f}_{weight_drum:.2f}".replace('.', '')
    metrics_out = OUT_DIR / f'eval_metrics_{tag}.json'
    rows_out = OUT_DIR / f'eval_rows_{tag}.csv'

    cmd = [
        sys.executable,
        str(EVAL_SCRIPT),
        '--folder', str(AUDIO_FOLDER),
        '--model_a', str(MODEL_A),
        '--features_a', str(FEATURES_A),
        '--bars', BARS,
        '--top_k', str(TOP_K),
        '--weight_model_score', str(weight_model),
        '--weight_periodicity', str(weight_periodicity),
        '--weight_drum_alignment', str(weight_drum),
        '--rows_out', str(rows_out),
        '--metrics_out', str(metrics_out),
    ]

    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)
    return metrics_out


def parse_metrics(metrics_path: Path) -> dict:
    payload = json.loads(metrics_path.read_text(encoding='utf-8'))
    weights = payload['composite_eval']['weights_requested']
    deltas = payload['model_a'].get('composite_vs_raw', {})
    d_topk = float(deltas.get('delta_topk_target', 0.0))
    d_ndcg = float(deltas.get('delta_ndcg_at_k', 0.0))
    return {
        'metrics_file': str(metrics_path),
        'model_score': float(weights['model_score']),
        'periodicity': float(weights['periodicity']),
        'drum_alignment': float(weights['drum_alignment']),
        'delta_topk_target': d_topk,
        'delta_ndcg_at_k': d_ndcg,
        'composite_uplift': d_topk + d_ndcg,
    }

In [ ]:
# Run all requested sweeps
run_ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
rows = []
for wm, wp, wd in WEIGHT_CONFIGS:
    metrics_file = run_one_sweep(wm, wp, wd)
    r = parse_metrics(metrics_file)
    r['run_timestamp'] = run_ts
    r['run_notes'] = RUN_NOTE
    rows.append(r)

run_df = pd.DataFrame(rows).sort_values('composite_uplift', ascending=False).reset_index(drop=True)
run_df['rank_in_run'] = run_df['composite_uplift'].rank(method='dense', ascending=False).astype(int)

# Make run_notes the final column as requested
ordered_cols = [
    'run_timestamp',
    'rank_in_run',
    'model_score',
    'periodicity',
    'drum_alignment',
    'delta_topk_target',
    'delta_ndcg_at_k',
    'composite_uplift',
    'metrics_file',
    'run_notes',
]
run_df = run_df[ordered_cols]

display(run_df)

In [ ]:
# Append this run to all-time leaderboard and keep winner ranking updated
if LEADERBOARD_CSV.exists():
    all_df = pd.read_csv(LEADERBOARD_CSV)
else:
    all_df = pd.DataFrame(columns=run_df.columns.tolist())

all_df = pd.concat([all_df, run_df], ignore_index=True)
all_df['winner_rank_all_time'] = all_df['composite_uplift'].rank(method='dense', ascending=False).astype(int)

# Ensure run_notes is final column
cols = [c for c in all_df.columns if c != 'run_notes'] + ['run_notes']
all_df = all_df[cols]

all_df.to_csv(LEADERBOARD_CSV, index=False)
print(f'Saved leaderboard: {LEADERBOARD_CSV}')
display(all_df.sort_values(['winner_rank_all_time', 'run_timestamp']).head(20))

In [ ]:
# Optional: update note for latest run after review
LATEST_NOTE = ''  # e.g., 'kept winner as weekly default after listening pass'

if LATEST_NOTE.strip():
    lb = pd.read_csv(LEADERBOARD_CSV)
    latest_ts = lb['run_timestamp'].max()
    lb.loc[lb['run_timestamp'] == latest_ts, 'run_notes'] = LATEST_NOTE.strip()
    cols = [c for c in lb.columns if c != 'run_notes'] + ['run_notes']
    lb = lb[cols]
    lb.to_csv(LEADERBOARD_CSV, index=False)
    print('Updated note for latest run batch.')
else:
    print('No note update requested.')

In [ ]:
# Optional: pull supporting references from Master Brain API (matches your PowerShell call style)
import json
import urllib.request
from pathlib import Path

MB_URL = 'http://127.0.0.1:8787/v1/query'
MB_API_KEY = 'master-brain-bridge-local'
MB_PROJECT_ROOT = 'C:/Users/dunnm/Downloads/Master-Brain-API'

query_payload = {
    'question': (
        'Provide authoritative sources and concise explanations for NDCG@k, top-k ranking quality metrics, '
        'weighted linear score fusion for ranking, and practical evaluation considerations for music/audio retrieval pipelines. '
        'Return titles, authors, year, venue, and URLs where possible.'
    ),
    'k': 10,
    'project_root': MB_PROJECT_ROOT,
}

req = urllib.request.Request(
    MB_URL,
    data=json.dumps(query_payload).encode('utf-8'),
    headers={
        'Content-Type': 'application/json',
        'x-api-key': MB_API_KEY,
        'x-project-root': MB_PROJECT_ROOT,
    },
    method='POST',
)

with urllib.request.urlopen(req, timeout=60) as resp:
    body = json.loads(resp.read().decode('utf-8'))

out_json = OUT_DIR / 'master_brain_sources.json'
out_json.write_text(json.dumps(body, indent=2), encoding='utf-8')

print('Saved:', out_json)
print('\nAnswer:\n', body.get('answer', ''))
print('\nTop context snippets:')
for c in body.get('context', [])[:10]:
    print('-', c)

print('\nTip: if context is off-topic, tighten the question or point MB_PROJECT_ROOT to a ranking/IR-focused index.')

## References and citations
Below are canonical references for the metrics and ranking logic used in this notebook.

1. **Järvelin, K., & Kekäläinen, J. (2002).** Cumulated gain-based evaluation of IR techniques. *ACM TOIS, 20(4), 422–446*. https://doi.org/10.1145/582415.582418
2. **Manning, C. D., Raghavan, P., & Schütze, H. (2008).** *Introduction to Information Retrieval* (ranking evaluation chapter). https://nlp.stanford.edu/IR-book/
3. **Liu, T.-Y. (2009).** Learning to Rank for Information Retrieval. *Foundations and Trends in IR*. https://www.nowpublishers.com/article/Details/INR-016
4. **TREC Evaluation Tracks** (practical ranking evaluation conventions). https://trec.nist.gov/
5. **Master Brain Bridge API docs** (query endpoint and response schema used for runtime citation retrieval). http://127.0.0.1:8787/docs

### Master Brain integration note
Use the previous code cell to fetch project-specific supporting snippets from your Master Brain index and save them to `master_brain_sources_runtime.json`.